# Notebook para estimar generación eólica y crear sus variables lag

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

Salimos de la carpeta de `/notebooks` para obtener las funciones de `build.py`.

In [2]:
sys.path.insert(0, '../src')
from energy_dispatch.features.build import add_lag_features

Partimos de las lecturas limpias del paso 0 y de las tres estaciones cercanas seleccionadas en el paso 1. Con su viento ponderado por distancia estimamos la generación del parque Vader Piet y después creamos los retardos por estación. Así este notebook no depende de una etapa de ingesta adicional.

In [3]:
ROOT = Path('..').resolve()
READINGS_PATH = ROOT / 'data' / 'processed' / '0_cleaning' / 'aruba_readings_cleaned.json'
STATIONS_PATH = ROOT / 'data' / 'processed' / '1_station_filtering' / 'aruba_stations_ordered_by_distance_m.json'
CSV_OUTPUT_PATH = '../data/processed/4_eolic_lag_features/aruba_readings_with_eolic_lags.csv'
JSON_OUTPUT_PATH = '../data/processed/4_eolic_lag_features/aruba_readings_with_eolic_lags.json'
REQUIRED_COLUMNS = {'station_id', 'timestamp', 'wind_speed_kmh'}

df = pd.read_json(READINGS_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
missing_columns = REQUIRED_COLUMNS.difference(df.columns)
if missing_columns:
    raise KeyError(f'Faltan columnas requeridas: {sorted(missing_columns)}')

nearby_stations = pd.read_json(STATIONS_PATH)
weights = 1 / nearby_stations.set_index('id')['distance_m']
weather = df[df['station_id'].isin(weights.index)].copy()
weather['weight'] = weather['station_id'].map(weights)
wind_at_park = (
    weather.assign(weighted_wind=weather['wind_speed_kmh'] * weather['weight'])
    .groupby('timestamp')[['weighted_wind', 'weight']].sum()
    .eval('weighted_wind / weight')
)

# Curva simplificada Vestas V90-3.0 MW: 10 turbinas, buje a 80 m y 10 % de pérdidas.
wind_hub_ms = wind_at_park.div(3.6) * (80 / 10) ** 0.14
curve_ms = np.array([0, 4, 6, 8, 10, 12, 15, 25])
curve_kw = np.array([0, 97, 414, 944, 1668, 2447, 3000, 3000])
power_per_turbine = np.interp(wind_hub_ms, curve_ms, curve_kw)
power_per_turbine[(wind_hub_ms < 4) | (wind_hub_ms >= 25)] = 0
generation = pd.Series(power_per_turbine * 10 * 0.90, index=wind_at_park.index, name='generacion_eolica_kw')
df = df.merge(generation, left_on='timestamp', right_index=True, how='left', validate='many_to_one')

df[['station_id', 'timestamp', 'generacion_eolica_kw']].head()

,station_id,timestamp,generacion_eolica_kw
0,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:00:00+00:00,10590.681075
1,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:05:00+00:00,10741.300895
2,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:10:00+00:00,10866.533844
3,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:15:00+00:00,10951.828362
4,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:20:00+00:00,10987.616288


## Comprobación de continuidad temporal

Un *lag* de un paso equivale a cinco minutos, la frecuencia del conjunto de entrada. Antes de crear las variables comprobamos los saltos entre lecturas consecutivas de cada estación. Si falta una lectura, no se puede asumir que la lectura anterior represente exactamente el instante requerido por el retardo.

In [4]:
ordered = df.sort_values(['station_id', 'timestamp']).copy()
ordered['delta_t'] = ordered.groupby('station_id')['timestamp'].diff()

gaps = ordered[ordered['delta_t'] > pd.Timedelta('5min')]
print(f'Lecturas con un hueco previo: {len(gaps):,}')
gaps[['station_id', 'timestamp', 'delta_t']].head()

Lecturas con un hueco previo: 36


,station_id,timestamp,delta_t
36000,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-01-11 00:00:00+00:00,7 days 00:05:00
58728,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-03-31 22:00:00+00:00,1 days 00:05:00
72288,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-05-25 00:00:00+00:00,7 days 00:05:00
141672,4ca4c103-894d-4f59-a911-f3abb8077e3c,2026-01-11 00:00:00+00:00,7 days 00:05:00
164400,4ca4c103-894d-4f59-a911-f3abb8077e3c,2026-03-31 22:00:00+00:00,1 days 00:05:00


## Creación de los retardos

`add_lag_features` ordena por estación y tiempo, y aplica `shift` positivo. Para una fila en el instante *t*, las nuevas columnas contienen:

- `generacion_eolica_kw_lag_1`: potencia en *t - 5 min*.
- `generacion_eolica_kw_lag_2`: potencia en *t - 10 min*.
- `generacion_eolica_kw_lag_3`: potencia en *t - 15 min*.
- `generacion_eolica_kw_lag_4`: potencia en *t - 20 min*.

Además de desplazar el valor, valida que la diferencia entre timestamps sea exactamente el número de pasos esperado. Por eso las primeras filas de cada estación y las posteriores a un hueco quedan como `NaN`.

In [5]:
df_with_lags = add_lag_features(
    df,
    target_col='generacion_eolica_kw',
    lags=range(1, 5),
    group_col='station_id',
    timestamp_col='timestamp',
    freq='5min',
)

lag_columns = [f'generacion_eolica_kw_lag_{lag}' for lag in range(1, 5)]
df_with_lags[['station_id', 'timestamp', 'generacion_eolica_kw', *lag_columns]].head(10)

,station_id,timestamp,generacion_eolica_kw,generacion_eolica_kw_lag_1,generacion_eolica_kw_lag_2,generacion_eolica_kw_lag_3,generacion_eolica_kw_lag_4
0,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:00:00+00:00,10590.681075,NaN,NaN,NaN,NaN
1,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:05:00+00:00,10741.300895,10590.681075,NaN,NaN,NaN
2,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:10:00+00:00,10866.533844,10741.300895,10590.681075,NaN,NaN
3,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:15:00+00:00,10951.828362,10866.533844,10741.300895,10590.681075,NaN
4,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:20:00+00:00,10987.616288,10951.828362,10866.533844,10741.300895,10590.681075
5,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:25:00+00:00,11122.417398,10987.616288,10951.828362,10866.533844,10741.300895
6,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:30:00+00:00,11204.586641,11122.417398,10987.616288,10951.828362,10866.533844
7,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:35:00+00:00,11223.530626,11204.586641,11122.417398,10987.616288,10951.828362
8,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:40:00+00:00,11468.820788,11223.530626,11204.586641,11122.417398,10987.616288
9,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:45:00+00:00,11631.941707,11468.820788,11223.530626,11204.586641,11122.417398


Contamos los valores ausentes que se han generado de forma intencionada. No se rellenan: inventar un valor en un hueco temporal convertiría una observación antigua en un supuesto dato reciente.

In [6]:
missing_lags = df_with_lags[lag_columns].isna().sum().rename('missing_values')
missing_lags.to_frame()

,missing_values
generacion_eolica_kw_lag_1,48
generacion_eolica_kw_lag_2,96
generacion_eolica_kw_lag_3,144
generacion_eolica_kw_lag_4,192


## Datos listos para el modelo

Para entrenar un modelo que requiera todas las variables *lag*, eliminamos únicamente las filas que no tienen alguno de esos retardos. Las columnas meteorológicas y el objetivo se conservan para seleccionar después las variables definitivas del modelo.

In [7]:
model_df = df_with_lags.dropna(subset=lag_columns).copy()
print(f'Filas originales: {len(df_with_lags):,}')
print(f'Filas con todos los lags: {len(model_df):,}')

model_df[['station_id', 'timestamp', 'generacion_eolica_kw', *lag_columns]].head()

Filas originales: 1,268,064
Filas con todos los lags: 1,267,872


,station_id,timestamp,generacion_eolica_kw,generacion_eolica_kw_lag_1,generacion_eolica_kw_lag_2,generacion_eolica_kw_lag_3,generacion_eolica_kw_lag_4
4,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:20:00+00:00,10987.616288,10951.828362,10866.533844,10741.300895,10590.681075
5,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:25:00+00:00,11122.417398,10987.616288,10951.828362,10866.533844,10741.300895
6,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:30:00+00:00,11204.586641,11122.417398,10987.616288,10951.828362,10866.533844
7,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:35:00+00:00,11223.530626,11204.586641,11122.417398,10987.616288,10951.828362
8,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:40:00+00:00,11468.820788,11223.530626,11204.586641,11122.417398,10987.616288


## Exportación de variables para el modelo

Guardamos `model_df` sin modificar la entrada. La exportación conserva las columnas originales y añade los cuatro retardos; solo incluye filas cuyos retardos son continuos y están disponibles.

In [8]:
model_df.to_csv(CSV_OUTPUT_PATH, index=False)
model_df.to_json(JSON_OUTPUT_PATH, orient='records', indent=2, force_ascii=False, date_format='iso')

print(f'Tabla para el modelo exportada en: {CSV_OUTPUT_PATH}')
print(f'Tabla para el modelo exportada en: {JSON_OUTPUT_PATH}')

Tabla para el modelo exportada en: ../data/processed/4_eolic_lag_features/aruba_readings_with_eolic_lags.csv
Tabla para el modelo exportada en: ../data/processed/4_eolic_lag_features/aruba_readings_with_eolic_lags.json
